In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
df = pd.read_csv("Cleaned_IT_Incident_Data.csv")
print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


Dataset loaded successfully!
Rows: 1000
Columns: 25


In [4]:
df.head()


,incident_id,timestamp,server_id,department,environment,server_type,cpu_usage_pct,memory_usage_pct,disk_usage_pct,network_latency_ms,...,uptime_days,previous_incidents_30d,maintenance_due,anomaly_score,incident_occurred,severity,incident_type,status,resolution_time_minutes,root_cause
0,INC-0001,2025-01-01 07:05:00,SRV-008,Customer Support,Staging,Web Server,23.32,55.73,66.22,103.10,...,1,0,No,0.00,No,No Incident,No Incident,Normal,0.0,No Incident
1,INC-0002,2025-01-01 09:59:00,SRV-010,Customer Support,Production,File Server,82.30,72.72,59.27,138.25,...,141,1,No,5.61,No,No Incident,No Incident,Normal,0.0,No Incident
2,INC-0003,2025-01-01 20:06:00,SRV-043,Operations,Production,API Gateway,74.19,64.79,54.89,165.19,...,25,2,Yes,49.50,No,No Incident,No Incident,Normal,0.0,No Incident
3,INC-0004,2025-01-02 05:33:00,SRV-050,Operations,Development,Database Server,22.94,54.27,42.49,46.62,...,60,1,Yes,6.83,No,No Incident,No Incident,Normal,0.0,No Incident
4,INC-0005,2025-01-02 13:40:00,SRV-024,IT,Production,Application Server,48.15,39.90,79.66,65.56,...,41,0,No,11.10,No,No Incident,No Incident,Normal,0.0,No Incident


In [5]:
print(df.columns.tolist())

['incident_id', 'timestamp', 'server_id', 'department', 'environment', 'server_type', 'cpu_usage_pct', 'memory_usage_pct', 'disk_usage_pct', 'network_latency_ms', 'response_time_ms', 'error_count', 'active_users', 'packet_loss_pct', 'temperature_c', 'uptime_days', 'previous_incidents_30d', 'maintenance_due', 'anomaly_score', 'incident_occurred', 'severity', 'incident_type', 'status', 'resolution_time_minutes', 'root_cause']


In [6]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing values: 0
Duplicate rows: 0


In [7]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)
df["month"] = df["timestamp"].dt.month
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
print("Date features created!")

Date features created!


In [8]:
df[[
    "timestamp",
    "month",
    "hour",
    "day_of_week"
]].head()

,timestamp,month,hour,day_of_week
0,2025-01-01 07:05:00,1,7,2
1,2025-01-01 09:59:00,1,9,2
2,2025-01-01 20:06:00,1,20,2
3,2025-01-02 05:33:00,1,5,3
4,2025-01-02 13:40:00,1,13,3


In [9]:
df["average_resource_usage"] = (
    df["cpu_usage_pct"] +
    df["memory_usage_pct"] +
    df["disk_usage_pct"]
) / 3

In [10]:
df["average_resource_usage"]

0      48.423333
1      71.430000
2      64.623333
3      39.900000
4      55.903333
         ...    
995    47.513333
996    50.456667
997    58.236667
998    53.960000
999    84.766667
Name: average_resource_usage, Length: 1000, dtype: float64

In [11]:
df[[
    "cpu_usage_pct",
    "memory_usage_pct",
    "disk_usage_pct",
    "average_resource_usage"
]].head()


,cpu_usage_pct,memory_usage_pct,disk_usage_pct,average_resource_usage
0,23.32,55.73,66.22,48.423333
1,82.30,72.72,59.27,71.430000
2,74.19,64.79,54.89,64.623333
3,22.94,54.27,42.49,39.900000
4,48.15,39.90,79.66,55.903333


In [14]:
df["error_rate_per_100_users"] = (
    df["error_count"] /
    (df["active_users"] + 1)
) * 100


In [15]:
df[[
    "error_count",
    "active_users",
    "error_rate_per_100_users"
]].head()

,error_count,active_users,error_rate_per_100_users
0,0,176,0.000000
1,11,349,3.142857
2,15,219,6.818182
3,0,90,0.000000
4,0,174,0.000000


In [17]:
df["high_utilization"] = np.where(
    (df["cpu_usage_pct"] >= 80) |
    (df["memory_usage_pct"] >= 80) |
    (df["disk_usage_pct"] >= 85),
    1,
    0
)

In [18]:
df["high_utilization"].value_counts()

high_utilization
0    828
1    172
Name: count, dtype: int64

In [20]:
df["incident_target"] = df["incident_occurred"].map({
    "No": 0,
    "Yes": 1
})


In [21]:
print(df[[
    "incident_occurred",
    "incident_target"
]].head())
print(df["incident_target"].value_counts())

  incident_occurred  incident_target
0                No                0
1                No                0
2                No                0
3                No                0
4                No                0
incident_target
0    902
1     98
Name: count, dtype: int64


In [22]:
drop_columns = [
    "incident_id",
    "timestamp",
    "incident_occurred",
    "incident_target",
    "severity",
    "incident_type",
    "status",
    "resolution_time_minutes",
    "root_cause"
]


In [30]:
drop_columns

['incident_id',
 'timestamp',
 'incident_occurred',
 'incident_target',
 'severity',
 'incident_type',
 'status',
 'resolution_time_minutes',
 'root_cause']

In [32]:
X = df.drop(columns=drop_columns)

print(X.head())

  server_id        department  environment         server_type  cpu_usage_pct  \
0   SRV-008  Customer Support      Staging          Web Server          23.32   
1   SRV-010  Customer Support   Production         File Server          82.30   
2   SRV-043        Operations   Production         API Gateway          74.19   
3   SRV-050        Operations  Development     Database Server          22.94   
4   SRV-024                IT   Production  Application Server          48.15   

   memory_usage_pct  disk_usage_pct  network_latency_ms  response_time_ms  \
0             55.73           66.22              103.10            462.70   
1             72.72           59.27              138.25            989.68   
2             64.79           54.89              165.19            925.91   
3             54.27           42.49               46.62            288.79   
4             39.90           79.66               65.56            392.74   

   error_count  ...  uptime_days  previous_inciden

In [40]:
categorical_columns = X.select_dtypes(
    include=["object","string"]
).columns
print("Categorical columns:")
print(categorical_columns.tolist())


Categorical columns:
['server_id', 'department', 'environment', 'server_type', 'maintenance_due']


In [41]:
X_encoded = pd.get_dummies(
    X,
    drop_first=True
)
print("Before encoding:", X.shape)
print("After encoding:", X_encoded.shape)

Before encoding: (1000, 23)
After encoding: (1000, 79)


In [42]:
print(
    "Missing values:",
    X_encoded.isnull().sum().sum()
)


Missing values: 0


In [46]:
y = df["incident_target"]

print(y.head())
print(y.value_counts())

0    0
1    0
2    0
3    0
4    0
Name: incident_target, dtype: int64
incident_target
0    902
1     98
Name: count, dtype: int64


In [47]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [50]:
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Training target rows:", Y_train.shape[0])
print("Testing target rows:", Y_test.shape[0])

Training rows: 800
Testing rows: 200
Training target rows: 800
Testing target rows: 200


In [52]:
print("Training target:")
print(Y_train.value_counts())
print("Testing target:")
print(Y_test.value_counts())

Training target:
incident_target
0    722
1     78
Name: count, dtype: int64
Testing target:
incident_target
0    180
1     20
Name: count, dtype: int64


In [61]:
print("Training incident percentage:", round(Y_train.mean() * 100, 2), "%")

print("Testing incident percentage:", round(Y_test.mean() * 100, 2), "%")

Training incident percentage: 9.75 %
Testing incident percentage: 10.0 %


In [62]:
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
Y_train.to_csv("y_train.csv", index=False)
Y_test.to_csv("y_test.csv", index=False)
print("Training and testing files saved successfully!")

Training and testing files saved successfully!


In [65]:
print("=" * 55)
print("PHASE 3 - FEATURE ENGINEERING SUMMARY")
# AI-Powered IT Incident Prediction & Prevention System
print("=" * 55)
print("Total records:", len(df))
print("Input features:", X_encoded.shape[1])
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training incidents:", Y_train.sum())
print("Testing incidents:", Y_test.sum())
print("Missing values:", X_encoded.isnull().sum().sum())
print("=" * 55)
print("Phase 3 completed successfully!")

PHASE 3 - FEATURE ENGINEERING SUMMARY
Total records: 1000
Input features: 79
Training records: 800
Testing records: 200
Training incidents: 78
Testing incidents: 20
Missing values: 0
Phase 3 completed successfully!


# Feature engineering and machine-learning preparation are complete. New features were created for 
resource usage, error rate, high utilization, month, hour and day of week. The incident target was 
converted from Yes/No to 1/0. Post-incident columns were removed to prevent data leakage. Text 
categories were converted into numeric columns using one-hot encoding. Finally, the data was divided 
into 80% training data and 20% testing data using stratified sampling.